# Veille Technique – SetFit : Few-Shot Learning pour la Classification de Texte

## Contexte
Dans le cadre du Projet 6 (classification automatique de produits e-commerce Flipkart), nous avions évalué 4 approches NLP classiques :
- Bag-of-Words → ARI = 0.245
- TF-IDF (bigrams) → ARI = 0.268  
- Word2Vec (GloVe) → ARI = 0.291
- USE (Universal Sentence Encoder) → ARI = 0.355

Ce notebook implémente **SetFit (2022)**, une technique récente de few-shot learning basée sur les Sentence Transformers, et la compare à ces approches classiques.

## Référence bibliographique
**Tunstall et al. (2022)** – *Efficient Few-Shot Learning Without Prompts*  
ArXiv : https://arxiv.org/abs/2209.11055  
Hugging Face Blog : https://huggingface.co/blog/setfit  
GitHub officiel : https://github.com/huggingface/setfit

## 1. Principe de SetFit – État de l'art

### 1.1 Problématique adressée
Les approches classiques (BoW, TF-IDF) nécessitent des milliers d'exemples labellisés pour être performantes. Le fine-tuning de BERT complet est coûteux en calcul et en données. SetFit répond à la question : **comment obtenir de bonnes performances avec très peu d'exemples labellisés ?**

### 1.2 Architecture
SetFit repose sur deux étapes :

**Étape 1 – Fine-tuning du Sentence Transformer par contrastive learning**

À partir d'un petit nombre d'exemples par classe (typiquement 8 à 64), SetFit génère des paires de phrases :
- Paires positives : deux phrases de la même classe → doivent être proches dans l'espace d'embedding
- Paires négatives : deux phrases de classes différentes → doivent être éloignées

La fonction de perte utilisée est la **Cosine Similarity Loss** :

$$\mathcal{L} = \sum_{(i,j)} \left( \cos(\mathbf{e}_i, \mathbf{e}_j) - y_{ij} \right)^2$$

où $y_{ij} = 1$ si même classe, $y_{ij} = 0$ sinon, et $\cos(\mathbf{e}_i, \mathbf{e}_j) = \frac{\mathbf{e}_i \cdot \mathbf{e}_j}{\|\mathbf{e}_i\| \|\mathbf{e}_j\|}$

**Étape 2 – Entraînement d'un classifieur léger**

Les embeddings fine-tunés sont utilisés comme features pour entraîner une tête de classification (par défaut Logistic Regression). Le classifieur apprend :

$$P(y=k | \mathbf{e}) = \text{softmax}(\mathbf{W} \mathbf{e} + \mathbf{b})_k$$

### 1.3 Avantages clés
- **Pas de prompt engineering** contrairement à GPT-few-shot
- **Très rapide** : fine-tuning en quelques minutes sur CPU
- **Efficace avec peu de données** : bonnes performances avec 8-64 exemples par classe
- **Flexible** : fonctionne avec n'importe quel Sentence Transformer de base

### 1.4 Comparaison avec USE (P6)
USE génère des embeddings fixes sans adaptation aux données. SetFit **adapte** le modèle d'embedding aux données cibles via le contrastive learning, ce qui lui permet de mieux capturer les nuances sémantiques spécifiques au domaine.

## 2. Installation et imports

In [ ]:
# installation des dépendances (à lancer une fois)
!pip install setfit sentence-transformers datasets -q


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import time
import re
import string

# nlp classique
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# setfit
from setfit import SetFitModel, Trainer, TrainingArguments
from datasets import Dataset

# sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    adjusted_rand_score, accuracy_score,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder
from sklearn.manifold import TSNE

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')


## 3. Chargement et préparation des données

In [ ]:
# chargement du dataset flipkart (même jeu qu'au p6)
df = pd.read_csv('data/Flipkart/data.csv')

# catégorie principale = premier niveau de l'arbre de catégories
def extract_main_category(category_tree):
    if pd.isna(category_tree):
        return None
    try:
        cat = str(category_tree).replace('["', '').replace('"]', '')
        return cat.split(' >> ')[0]
    except:
        return None

df['main_category'] = df['product_category_tree'].apply(extract_main_category)
df = df.dropna(subset=['main_category', 'product_name', 'description'])

print(f"Nombre de produits : {len(df)}")
print(f"Nombre de catégories : {df['main_category'].nunique()}")
print(df['main_category'].value_counts())


In [ ]:
# preprocessing texte identique au p6 (lemmatisation, stopwords, ponctuation)
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df['text_combined'] = df['product_name'].fillna('') + ' ' + df['description'].fillna('')
df['text_lemmatized'] = df['text_combined'].apply(preprocess_text)

print(f"Longueur moyenne : {df['text_lemmatized'].str.split().str.len().mean():.0f} mots")


In [ ]:
# encodage des labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['main_category'])

for i, cls in enumerate(le.classes_):
    print(f"{i} : {cls}")


## 4. Approche classique – TF-IDF + Logistic Regression (baseline P6)

On reproduit ici l'approche TF-IDF du P6 en mode **supervisé** (classification, pas clustering) pour avoir une baseline comparable à SetFit.

In [ ]:
# split train/test stratifié
X = df['text_lemmatized'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {len(X_train)} exemples")
print(f"Test  : {len(X_test)} exemples")


In [ ]:
# baseline : tf-idf + régression logistique (supervisé, tout le train)
t0 = time.time()

tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.8,
    ngram_range=(1, 2),
    sublinear_tf=True
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

lr_tfidf = LogisticRegression(max_iter=1000, random_state=42)
lr_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf = lr_tfidf.predict(X_test_tfidf)

acc_tfidf = accuracy_score(y_test, y_pred_tfidf)
ari_tfidf = adjusted_rand_score(y_test, y_pred_tfidf)

print(f"Temps : {time.time() - t0:.1f}s")
print(f"Accuracy : {acc_tfidf:.4f}")
print(f"ARI : {ari_tfidf:.4f}")
print(classification_report(y_test, y_pred_tfidf, target_names=le.classes_))


## 5. SetFit – Few-Shot Learning

**Protocole few-shot** : on simule un scénario où on n'a que **16 exemples par catégorie** pour l'entraînement (= 112 exemples au total pour 7 classes), ce qui est le cas d'usage typique de SetFit.

In [ ]:
# dataset few-shot : 16 exemples par classe tirés aléatoirement
N_SHOTS = 16

df_train = pd.DataFrame({'text': X_train, 'label': y_train})
df_test  = pd.DataFrame({'text': X_test,  'label': y_test})

few_shot_samples = []
for label in np.unique(y_train):
    class_samples = df_train[df_train['label'] == label]
    n = min(N_SHOTS, len(class_samples))
    few_shot_samples.append(class_samples.sample(n, random_state=42))

df_few_shot = pd.concat(few_shot_samples).reset_index(drop=True)

print(f"Few-shot : {len(df_few_shot)} exemples ({N_SHOTS} par classe)")
print(f"Test : {len(df_test)} exemples")


In [ ]:
# format Dataset attendu par setfit
train_dataset = Dataset.from_pandas(df_few_shot[['text', 'label']])
test_dataset  = Dataset.from_pandas(df_test[['text', 'label']])


In [ ]:
# modèle de base : paraphrase-MiniLM-L6-v2 (léger ~80Mo, tourne sur cpu)
model = SetFitModel.from_pretrained(
    "sentence-transformers/paraphrase-MiniLM-L6-v2",
    num_setfit_iterations=20,
    num_epochs=1
)


In [ ]:
# fine-tuning setfit (contrastive learning sur le few-shot)
args = TrainingArguments(
    num_epochs=1,
    batch_size=16,
    num_iterations=20,
    seed=42,
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    metric="accuracy",
)

t0 = time.time()
trainer.train()
print(f"Temps d'entraînement : {time.time() - t0:.1f}s")


In [ ]:
# évaluation sur le test set
y_pred_setfit = model.predict(df_test['text'].tolist())
if hasattr(y_pred_setfit, 'numpy'):
    y_pred_setfit = y_pred_setfit.numpy()

acc_setfit = accuracy_score(y_test, y_pred_setfit)
ari_setfit = adjusted_rand_score(y_test, y_pred_setfit)

print(f"SetFit ({N_SHOTS} exemples/classe)")
print(f"Accuracy : {acc_setfit:.4f}")
print(f"ARI : {ari_setfit:.4f}")
print(classification_report(y_test, y_pred_setfit, target_names=le.classes_))


## 6. Comparaison des approches

In [ ]:
# tableau comparatif avec les méthodes du p6
results = pd.DataFrame({
    'Méthode': [
        'BoW + KMeans (P6)',
        'TF-IDF + KMeans (P6)',
        'Word2Vec + KMeans (P6)',
        'USE + KMeans (P6)',
        'TF-IDF + LogReg (supervisé, 100%)',
        f'SetFit (few-shot, {N_SHOTS}/classe)'
    ],
    'Type': [
        'Non-supervisé', 'Non-supervisé', 'Non-supervisé', 'Non-supervisé',
        'Supervisé (baseline)', 'Few-shot (SetFit)'
    ],
    'Données d\'entraînement': [
        '1050 (non labellisées)',
        '1050 (non labellisées)',
        '1050 (non labellisées)',
        '1050 (non labellisées)',
        f'{len(X_train)} (labellisées)',
        f'{N_SHOTS * df["main_category"].nunique()} (labellisées)'
    ],
    'ARI': [
        0.245, 0.268, 0.291, 0.355,
        round(ari_tfidf, 3),
        round(ari_setfit, 3)
    ],
    'Accuracy': [
        'N/A', 'N/A', 'N/A', 'N/A',
        round(acc_tfidf, 3),
        round(acc_setfit, 3)
    ]
})

print(results.to_string(index=False))


In [ ]:
# comparaison ari + positionnement few-shot de setfit
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# palette accessible daltonisme (okabe-ito)
colors = ['#56B4E9', '#56B4E9', '#56B4E9', '#56B4E9', '#E69F00', '#009E73']

methodes = results['Méthode'].tolist()
ari_vals = [0.245, 0.268, 0.291, 0.355, round(ari_tfidf, 3), round(ari_setfit, 3)]

bars = axes[0].barh(methodes, ari_vals, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('Adjusted Rand Index (ARI)', fontsize=11)
axes[0].set_title('Comparaison ARI – Toutes les méthodes', fontweight='bold', fontsize=12)
axes[0].axvline(x=0, color='black', linewidth=0.8)
for bar, val in zip(bars, ari_vals):
    axes[0].text(val + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)

legend_patches = [
    mpatches.Patch(color='#56B4E9', label='Non-supervisé (P6)'),
    mpatches.Patch(color='#E69F00', label='Supervisé TF-IDF (100% données)'),
    mpatches.Patch(color='#009E73', label=f'SetFit few-shot ({N_SHOTS}/classe)'),
]
axes[0].legend(handles=legend_patches, fontsize=9, loc='lower right')

# panneau droit : le point setfit few-shot face à la baseline supervisée
axes[1].set_title('Contexte few-shot : efficacité de SetFit\n(performance vs données labellisées)',
                  fontweight='bold', fontsize=11)
axes[1].axhline(y=acc_tfidf, color='#E69F00', linestyle='--', linewidth=2,
               label=f'TF-IDF supervisé (100% train = {len(X_train)} ex.) : {acc_tfidf:.3f}')
axes[1].scatter([N_SHOTS], [acc_setfit], color='#009E73', s=150, zorder=5,
               label=f'SetFit ({N_SHOTS} ex./classe) : {acc_setfit:.3f}', marker='*')
axes[1].set_xlabel('Nombre d\'exemples labellisés utilisés par SetFit', fontsize=10)
axes[1].set_ylabel('Accuracy', fontsize=10)
axes[1].set_xlim([0, 80])
axes[1].set_ylim([0, 1.05])
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('comparaison_setfit_p6.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# matrices de confusion en effectifs (tf-idf vs setfit)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title in zip(
    axes,
    [y_pred_tfidf, y_pred_setfit],
    [f'TF-IDF + LogReg (supervisé, {len(X_train)} exemples)',
     f'SetFit (few-shot, {N_SHOTS * df["main_category"].nunique()} exemples)']
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, ax=ax,
        annot=True, fmt='d',
        xticklabels=le.classes_,
        yticklabels=le.classes_,
        cmap='Blues'
    )
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('Prédiction')
    ax.set_ylabel('Réel')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('matrices_confusion_setfit.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# projection t-sne des embeddings setfit fine-tunés
embeddings = model.encode(df['text_lemmatized'].tolist())

t0 = time.time()
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000, verbose=0)
coords_2d = tsne.fit_transform(embeddings)
print(f"T-SNE : {time.time() - t0:.1f}s")

colors_tsne = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7']

fig, ax = plt.subplots(figsize=(10, 7))
for i, category in enumerate(le.classes_):
    mask = df['main_category'] == category
    ax.scatter(
        coords_2d[mask, 0], coords_2d[mask, 1],
        c=colors_tsne[i], label=category,
        alpha=0.7, s=30, edgecolors='none'
    )

ax.set_title('T-SNE – Embeddings SetFit fine-tunés\n(chaque couleur = une catégorie)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Composante T-SNE 1')
ax.set_ylabel('Composante T-SNE 2')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('tsne_setfit.png', dpi=150, bbox_inches='tight')
plt.show()


### Courbe d'apprentissage : efficacité en données

Performance de SetFit en fonction du nombre d'exemples labellisés par classe (8, 16, 32), moyennée sur 3 tirages aléatoires (seeds) pour estimer la variabilité. C'est la démonstration clé de l'efficacité en données : la performance monte vite puis se stabilise, et les barres d'erreur se resserrent quand le nombre d'exemples augmente.

In [ ]:
# courbe d'apprentissage : accuracy/ari selon le nombre d'exemples par classe
# résultats relus depuis le csv s'il existe (évite ~2h de recalcul)
import os, gc

CURVE_CSV  = 'courbe_learning_setfit.csv'
shots_grid = [8, 16, 32]

if os.path.exists(CURVE_CSV):
    curve_df = pd.read_csv(CURVE_CSV)
    print(f"Résultats relus depuis {CURVE_CSV} ({len(curve_df)} runs)")
else:
    seeds      = [42, 0, 1]   # plusieurs tirages pour estimer la variabilité
    BASE_MODEL = "sentence-transformers/paraphrase-MiniLM-L6-v2"
    n_classes_lc = len(np.unique(y_train))

    curve_rows = []
    n_runs = len(shots_grid) * len(seeds)
    run, t_start = 0, time.time()

    for n_shots in shots_grid:
        for seed in seeds:
            run += 1
            # sous-échantillonnage stratifié reproductible
            parts = []
            for label in np.unique(y_train):
                class_samples = df_train[df_train['label'] == label]
                k = min(n_shots, len(class_samples))
                parts.append(class_samples.sample(k, random_state=seed))
            df_fs = pd.concat(parts).reset_index(drop=True)
            train_ds = Dataset.from_pandas(df_fs[['text', 'label']])

            # modèle rechargé à neuf à chaque run (sinon on entraîne par-dessus le précédent)
            model_lc = SetFitModel.from_pretrained(BASE_MODEL)
            args_lc  = TrainingArguments(num_epochs=1, batch_size=16,
                                         num_iterations=20, seed=seed)
            trainer_lc = Trainer(model=model_lc, args=args_lc, train_dataset=train_ds)

            t0 = time.time()
            trainer_lc.train()
            dt = time.time() - t0

            y_hat = model_lc.predict(df_test['text'].tolist())
            if hasattr(y_hat, 'numpy'):
                y_hat = y_hat.numpy()

            curve_rows.append({
                'n_shots': n_shots, 'seed': seed,
                'n_total': n_shots * n_classes_lc,
                'accuracy': accuracy_score(y_test, y_hat),
                'ari': adjusted_rand_score(y_test, y_hat),
                'temps_s': dt,
            })

            elapsed = time.time() - t_start
            eta = elapsed / run * (n_runs - run)
            print(f"[{run}/{n_runs}] n_shots={n_shots:>2} seed={seed:>2} "
                  f"acc={curve_rows[-1]['accuracy']:.3f} ari={curve_rows[-1]['ari']:.3f} "
                  f"({dt:.0f}s, eta ~{eta/60:.0f}min)")

            pd.DataFrame(curve_rows).to_csv(CURVE_CSV, index=False)
            del model_lc, trainer_lc
            gc.collect()

    curve_df = pd.DataFrame(curve_rows)

curve_df


In [ ]:
# moyenne ± écart-type par taille de few-shot, puis tracé
agg = (curve_df.groupby('n_shots')
       .agg(acc_mean=('accuracy', 'mean'), acc_std=('accuracy', 'std'),
            ari_mean=('ari', 'mean'),      ari_std=('ari', 'std'))
       .reset_index())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (mean_col, std_col, base, base_lbl, color, ylabel, marker) in zip(axes, [
    ('acc_mean', 'acc_std', acc_tfidf, f'TF-IDF supervisé (840 ex.) : {acc_tfidf:.3f}', '#009E73', 'Accuracy (test)', 'o'),
    ('ari_mean', 'ari_std', ari_tfidf, f'TF-IDF supervisé (840 ex.) : {ari_tfidf:.3f}', '#0072B2', 'ARI (test)', 's'),
]):
    ax.errorbar(agg['n_shots'], agg[mean_col], yerr=agg[std_col],
                marker=marker, capsize=5, linewidth=2, color=color, label='SetFit few-shot')
    ax.axhline(base, ls='--', color='#E69F00', linewidth=2, label=base_lbl)
    ax.set_xlabel("Exemples labellisés par classe")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{ylabel.split(' ')[0]} vs taille du few-shot\n(moyenne ± écart-type sur 3 seeds)",
                 fontweight='bold', fontsize=11)
    ax.set_xticks(shots_grid)
    ax.set_ylim(0, 1)
    ax.legend()

plt.tight_layout()
plt.savefig('courbe_learning_setfit.png', dpi=150, bbox_inches='tight')
plt.show()

for _, r in agg.iterrows():
    print(f"{int(r['n_shots']):>2} ex./classe : acc = {r['acc_mean']:.3f} ± {r['acc_std']:.3f}")


## 7. Synthèse et conclusion

La preuve de concept compare **SetFit** (few-shot, sans prompt) à une baseline classique **TF-IDF + régression logistique** sur le dataset Flipkart (7 catégories), avec un jeu de test identique.

**Résultat principal.** Avec seulement 112 exemples labellisés (16 par classe), SetFit atteint 82,4 % d'accuracy (ARI 0,652), contre 95,2 % (ARI 0,893) pour TF-IDF entraîné sur 840 exemples. SetFit n'égale donc pas la baseline pleinement supervisée, mais il en atteint ~87 % de la performance avec **7,5× moins de données annotées**.

**Efficacité en données.** La courbe d'apprentissage confirme le vrai intérêt de la technique : la performance grimpe vite (78,4 % → 84,8 % → 87,9 % pour 8, 16 et 32 exemples/classe) et se stabilise (écart-type de ±0,040 à ±0,005). L'avantage de SetFit n'est pas d'être *plus performant*, mais d'être **quasi aussi performant avec très peu d'exemples**.

**Interprétabilité.** La projection T-SNE des embeddings fine-tunés montre des clusters bien séparés par catégorie, et les matrices de confusion identifient les confusions résiduelles entre catégories proches.

**Cas d'usage.** SetFit est pertinent lorsque l'annotation est coûteuse ou le domaine nouveau : il permet un prototype de qualité correcte à partir d'une poignée d'exemples, quitte à basculer vers une approche pleinement supervisée une fois plus de données disponibles.

In [ ]:
n_classes = df['main_category'].nunique()
n_few_shot_total = N_SHOTS * n_classes

print("Conclusions\n")
print(f"1. Avec {n_few_shot_total} exemples labellisés ({N_SHOTS}/classe), SetFit obtient "
      f"{acc_setfit:.1%} d'accuracy, contre {acc_tfidf:.1%} pour TF-IDF supervisé "
      f"entraîné sur {len(X_train)} exemples.\n")
print(f"2. SetFit n'égale pas la baseline supervisée, mais atteint environ "
      f"{acc_setfit/acc_tfidf:.0%} de sa performance avec {len(X_train)/n_few_shot_total:.1f}x "
      f"moins d'exemples labellisés.\n")
print("3. La courbe d'apprentissage (8, 16, 32 ex./classe) confirme ce point : la "
      "performance monte vite puis se stabilise, et la variabilité se resserre avec "
      "plus d'exemples.\n")
print("4. Le contrastive learning adapte les embeddings au domaine (produits "
      "e-commerce) sans fine-tuner tout le Transformer.\n")
print("Limites : résultats sensibles à l'échantillon few-shot (quantifié via la courbe), "
      "difficulté croissante avec le nombre de classes, inférence plus lente que TF-IDF.")
